## Import Libraries and Datasets

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
sorted(p.name for p in DATA_DIR.iterdir())

['bars_1m.parquet',
 'events.json',
 'exchange_calendar.csv',
 'regime_labels.csv',
 'sim_runs.json',
 'stream_bars.json',
 'symbol_history.csv',
 'symbols.csv',
 'vendor_daily.csv']

In [3]:
exchange_calendar = pd.read_csv(DATA_DIR / "exchange_calendar.csv")
symbol_history = pd.read_csv(DATA_DIR / "symbol_history.csv")
symbols = pd.read_csv(DATA_DIR / "symbols.csv")
vendor_daily = pd.read_csv(DATA_DIR / "vendor_daily.csv")
bars = pd.read_parquet(DATA_DIR / "bars_1m.parquet")

exchange_calendar["_source_row"] = range(len(exchange_calendar))
symbol_history["_source_row"] = range(len(symbol_history))
symbols["_source_row"] = range(len(symbols))
vendor_daily["_source_row"] = range(len(vendor_daily))
bars["_source_row"] = range(len(bars))

for name, df in [
    ("exchange_calendar", exchange_calendar),
    ("symbol_history", symbol_history),
    ("symbols", symbols),
    ("vendor_daily", vendor_daily),
    ("bars", bars),
]:
    print(f"{name:16s} {df.shape}  {list(df.columns)[:6]} ...")

exchange_calendar (129, 7)  ['date', 'is_trading_day', 'session_open', 'session_close', 'timezone', 'note'] ...
symbol_history   (3, 6)  ['security_id', 'change_type', 'old_value', 'new_value', 'effective_date', '_source_row'] ...
symbols          (31, 8)  ['security_id', 'ticker', 'company_name', 'sector', 'listing_date', 'delisting_date'] ...
vendor_daily     (10831, 9)  ['run_id', 'date', 'ticker', 'open', 'high', 'low'] ...
bars             (4276175, 10)  ['run_id', 'ticker', 'ts', 'open', 'high', 'low'] ...


## Flatten nested JSON

In [4]:
import json

with open(DATA_DIR / "sim_runs.json", encoding="utf-8") as f:
    sim_runs_raw = json.load(f)

with open(DATA_DIR / "events.json", encoding="utf-8") as f:
    events_raw = json.load(f)

In [5]:
sim_runs_flat = pd.json_normalize(sim_runs_raw, sep="_")
sim_runs_flat.insert(0, "_source_row", range(1, len(sim_runs_flat) + 1))

events_flat = pd.json_normalize(events_raw["events"], sep="_")
events_flat.insert(0, "_source_row", range(1, len(events_flat) + 1))

In [6]:
print("SIM_RUNS")
print(sim_runs_flat.shape)
print(sim_runs_flat.columns.tolist())
display(sim_runs_flat.head())

print("\nEVENTS")
print(events_flat.shape)
print(events_flat.columns.tolist())
display(events_flat.head())

SIM_RUNS
(4, 25)
['_source_row', 'run_id', 'config_version', 'created_at', 'seed', 'window_start', 'window_end', 'universe_stocks', 'universe_index_ticker', 'universe_index_method', 'universe_index_base_level', 'parameters_volatility_sigma_annual', 'parameters_climate_states', 'parameters_climate_variance_multiplier', 'parameters_climate_transition_matrix', 'parameters_news_memory_alpha', 'parameters_news_memory_beta', 'parameters_news_memory_gamma', 'parameters_intraday_open_boost', 'parameters_intraday_close_boost', 'parameters_shocks_distribution', 'parameters_shocks_df', 'parameters_jumps_rate_per_day', 'parameters_factors_market_weight', 'parameters_factors_sector_share']


,_source_row,run_id,config_version,created_at,seed,window_start,window_end,universe_stocks,universe_index_ticker,universe_index_method,...,parameters_news_memory_alpha,parameters_news_memory_beta,parameters_news_memory_gamma,parameters_intraday_open_boost,parameters_intraday_close_boost,parameters_shocks_distribution,parameters_shocks_df,parameters_jumps_rate_per_day,parameters_factors_market_weight,parameters_factors_sector_share
0,1,RUN-2025A,1.4.0,2025-07-02T08:00:00+02:00,8522640,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
1,2,RUN-2025B,1.4.1,2025-07-02T08:00:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
2,3,RUN-2025C,1.4.0,2025-07-02T08:00:00+02:00,8325121,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
3,4,RUN-2025B,1.4.1,2025-07-03T11:20:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25



EVENTS
(62, 8)
['_source_row', 'event_id', 'type', 'time', 'security_ticker', 'details_session', 'details_duration_minutes', 'details_ratio']


,_source_row,event_id,type,time,security_ticker,details_session,details_duration_minutes,details_ratio
0,1,EV0001,earnings,2025-01-10T09:30:00-05:00,DALN,pre_open,NaN,NaN
1,2,EV0002,earnings,2025-01-14T09:30:00-05:00,BRIS,pre_open,NaN,NaN
2,3,EV0003,earnings,2025-01-15T09:30:00-05:00,SOLK,pre_open,NaN,NaN
3,4,EV0004,earnings,2025-01-24T09:30:00-05:00,XERN,pre_open,NaN,NaN
4,5,EV0005,earnings,2025-01-30T09:30:00-05:00,VALX,pre_open,NaN,NaN


In [7]:
bars.head()

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0
1,RUN-2025A,ARVT,2025-01-02 09:31:00,119.93,119.98,119.35,119.39,45001,2025-01-02 14:32:01+00:00,1
2,RUN-2025A,ARVT,2025-01-02 09:32:00,119.39,119.66,119.30,119.53,5825,2025-01-02 14:33:00+00:00,2
3,RUN-2025A,ARVT,2025-01-02 09:33:00,119.53,119.57,119.23,119.35,8195,2025-01-02 14:34:00+00:00,3
4,RUN-2025A,ARVT,2025-01-02 09:34:00,119.35,119.44,119.33,119.38,5631,2025-01-02 14:35:01+00:00,4


In [8]:
sim_runs_flat.head()

,_source_row,run_id,config_version,created_at,seed,window_start,window_end,universe_stocks,universe_index_ticker,universe_index_method,...,parameters_news_memory_alpha,parameters_news_memory_beta,parameters_news_memory_gamma,parameters_intraday_open_boost,parameters_intraday_close_boost,parameters_shocks_distribution,parameters_shocks_df,parameters_jumps_rate_per_day,parameters_factors_market_weight,parameters_factors_sector_share
0,1,RUN-2025A,1.4.0,2025-07-02T08:00:00+02:00,8522640,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
1,2,RUN-2025B,1.4.1,2025-07-02T08:00:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
2,3,RUN-2025C,1.4.0,2025-07-02T08:00:00+02:00,8325121,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
3,4,RUN-2025B,1.4.1,2025-07-03T11:20:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25


## Functions

### Rejection Function

In [9]:
# Rejection 
def make_rejects(df, mask, reason, source_file):
    bad_rows = df.loc[mask]

    reject_rows = [
        {
            "raw_payload": row.drop(labels="_source_row").to_dict(),
            "source_file": source_file,
            "source_row": int(row["_source_row"]),
            "reason": reason,
        }
        for _, row in bad_rows.iterrows()
    ]

    return pd.DataFrame(reject_rows)


all_rejects = []


### Cleaning function (Upper Case & Stripping for String Columns)

In [10]:
def clean_text_values_check(df, columns=None):
    df_clean = df.copy()

    if columns is None:
        columns = df_clean.select_dtypes(include=["object", "string"]).columns

    df_clean[columns] = df_clean[columns].apply(
        lambda col: col.str.strip().str.upper()
    )

    return df_clean

### EDA `Symbols` 

In [11]:
symbols.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   security_id         31 non-null     str  
 1   ticker              31 non-null     str  
 2   company_name        31 non-null     str  
 3   sector              31 non-null     str  
 4   listing_date        31 non-null     str  
 5   delisting_date      1 non-null      str  
 6   shares_outstanding  31 non-null     int64
 7   _source_row         31 non-null     int64
dtypes: int64(2), str(6)
memory usage: 3.4 KB


In [12]:
symbols.head()

,security_id,ticker,company_name,sector,listing_date,delisting_date,shares_outstanding,_source_row
0,SEC001,ARVT,Arvon Systems,Technology,2019-01-03,NaN,772000000,0
1,SEC002,BELX,Belia Capital,Financials,2020-02-03,NaN,193000000,1
2,SEC003,CORB,Coron Therapeutics,Healthcare,2021-03-03,NaN,660000000,2
3,SEC004,DALN,Dalia Energy,Energy,2022-04-03,NaN,192000000,3
4,SEC005,ERTL,Erton Industries,Industrials,2023-05-03,NaN,141000000,4


In [13]:
symbols["security_id"].unique()

<ArrowStringArray>
['SEC001', 'SEC002', 'SEC003', 'SEC004', 'SEC005', 'SEC006', 'SEC007',
 'SEC008', 'SEC009', 'SEC010', 'SEC011', 'SEC012', 'SEC013', 'SEC014',
 'SEC015', 'SEC016', 'SEC017', 'SEC018', 'SEC019', 'SEC020', 'SEC021',
 'SEC022', 'SEC023', 'SEC024', 'SEC025', 'SEC026', 'SEC027', 'SEC028',
 'SEC029', 'SEC030']
Length: 30, dtype: str

I noticed here that the total entries are 31 while the unique values for the security ID are only 30 , so there must be a duplicate value.

In [14]:
duplicate_rows = symbols[symbols['security_id'].duplicated(keep=False)]

duplicate_rows.groupby('security_id').apply(
    lambda group: group.nunique(dropna=False).le(1).all()
)

security_id
SEC009    False
dtype: bool

The security ID that was duplicated is SEC009, So let's check if it has the same values in all columns or it is a different entry ?
First we need to make sure that all values have the same Format.

In [15]:
def clean_text_values(symbols):
    symbols_clean = symbols.copy()

    text_columns = symbols_clean.select_dtypes(include=['object', 'string']).columns

    symbols_clean[text_columns] = symbols_clean[text_columns].apply(
        lambda col: col.str.upper()
    )

    return symbols_clean

In [16]:
symbols_clean = clean_text_values(symbols)
symbols_clean.head()

,security_id,ticker,company_name,sector,listing_date,delisting_date,shares_outstanding,_source_row
0,SEC001,ARVT,ARVON SYSTEMS,TECHNOLOGY,2019-01-03,NaN,772000000,0
1,SEC002,BELX,BELIA CAPITAL,FINANCIALS,2020-02-03,NaN,193000000,1
2,SEC003,CORB,CORON THERAPEUTICS,HEALTHCARE,2021-03-03,NaN,660000000,2
3,SEC004,DALN,DALIA ENERGY,ENERGY,2022-04-03,NaN,192000000,3
4,SEC005,ERTL,ERTON INDUSTRIES,INDUSTRIALS,2023-05-03,NaN,141000000,4


In [17]:
duplicate_rows = symbols_clean[symbols_clean['security_id'].duplicated(keep=False)]

duplicate_rows.groupby('security_id').apply(
    lambda group: group.nunique(dropna=False).le(1).all()
)

security_id
SEC009    False
dtype: bool

In [18]:
duplicate_rows[duplicate_rows['security_id'] == 'SEC009']

,security_id,ticker,company_name,sector,listing_date,delisting_date,shares_outstanding,_source_row
8,SEC009,ISTK,ISTON THERAPEUTICS,HEALTHCARE,2022-09-03,NaN,320000000,8
30,SEC009,ISTK,ISTON THERAPEUTICS,HEALTHCARE,2022-09-03,NaN,320000000,30


After checking, The only difference between the two duplicate rows is the _source_row column, which is expected since it is a unique identifier for each row. All other columns have identical values, indicating that these rows are indeed duplicates in terms of their data content.     

In [19]:
#Removing Duplicates
symbols_clean = symbols_clean.drop_duplicates(subset='security_id', keep='first')

In [20]:
symbols_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   security_id         30 non-null     str  
 1   ticker              30 non-null     str  
 2   company_name        30 non-null     str  
 3   sector              30 non-null     str  
 4   listing_date        30 non-null     str  
 5   delisting_date      1 non-null      str  
 6   shares_outstanding  30 non-null     int64
 7   _source_row         30 non-null     int64
dtypes: int64(2), str(6)
memory usage: 3.3 KB


In [21]:
symbols_clean["delisting_date"].value_counts()

delisting_date
2025-04-30    1
Name: count, dtype: int64

We have also noticed that there is only one entry in the "delisting_date" variable, but we decided to keep it for further investigation with other Datasets.

In [22]:
symbols_clean.describe()

,shares_outstanding,_source_row
count,3.000000e+01,30.000000
mean,4.257333e+08,14.500000
std,2.512561e+08,8.803408
min,4.100000e+07,0.000000
25%,2.110000e+08,7.250000
50%,4.110000e+08,14.500000
75%,6.675000e+08,21.750000
max,8.910000e+08,29.000000


### EDA `Symbols History` 

In [23]:
symbol_history.head()

,security_id,change_type,old_value,new_value,effective_date,_source_row
0,SEC017,ticker,ORBT,ORBX,2025-03-17,0
1,SEC022,sector,Energy,Technology,2025-05-01,1
2,SEC010,shares_outstanding,207000000,414000000,2025-04-01,2


Now let's match the format of Symbols History with Symbols Dataset

In [24]:
def clean_text_values(symbols):
    History_cleaning = symbol_history.copy()

    text_columns = History_cleaning.select_dtypes(include=['object', 'string']).columns

    History_cleaning[text_columns] = History_cleaning[text_columns].apply(
        lambda col: col.str.upper()
    )

    return History_cleaning

In [25]:
History_cleaning = clean_text_values(symbol_history)
History_cleaning.head()

,security_id,change_type,old_value,new_value,effective_date,_source_row
0,SEC017,TICKER,ORBT,ORBX,2025-03-17,0
1,SEC022,SECTOR,ENERGY,TECHNOLOGY,2025-05-01,1
2,SEC010,SHARES_OUTSTANDING,207000000,414000000,2025-04-01,2


In [26]:
History_cleaning.columns

Index(['security_id', 'change_type', 'old_value', 'new_value',
       'effective_date', '_source_row'],
      dtype='str')

In [27]:
History_cleaning.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   security_id     3 non-null      str  
 1   change_type     3 non-null      str  
 2   old_value       3 non-null      str  
 3   new_value       3 non-null      str  
 4   effective_date  3 non-null      str  
 5   _source_row     3 non-null      int64
dtypes: int64(1), str(5)
memory usage: 401.0 bytes


We can see that the Symbols History dataset includes only 3 entries about 3 different companies, in which there is a change happened.

### EDA `bar_1m`

In [28]:
bars_cleaned = bars.copy()

In [29]:
bars.describe().style.format("{:,.2f}")

,open,high,low,close,volume,_source_row
count,"4,276,175.00","4,276,175.00","4,276,175.00","4,275,321.00","4,276,175.00","4,276,175.00"
mean,149.36,149.51,149.21,149.36,"8,358.85","2,138,087.00"
std,183.48,183.60,183.36,183.48,"25,639.99","1,234,425.54"
min,4.81,4.84,4.80,4.81,"-47,568.00",0.00
25%,63.54,63.59,63.48,63.54,"1,599.00","1,069,043.50"
50%,113.11,113.25,112.97,113.11,"2,825.00","2,138,087.00"
75%,157.54,157.75,157.35,157.54,"5,232.00","3,207,130.50"
max,"1,483.31","1,483.84","1,482.76","1,483.31","13,208,790.00","4,276,174.00"


#### Missing Values

In [30]:
bars.isna().sum()

run_id           0
ticker           0
ts               0
open             0
high             0
low              0
close          854
volume           0
received_at      0
_source_row      0
dtype: int64

In [31]:
mask = bars["close"].isna()
all_rejects.append(make_rejects(bars, mask, "MISSING_REQUIRED_FIELD", "bars_1m.parquet"))
bars_cleaned = bars.loc[~mask]

##### Negative Values

In [32]:
#-ve values
negative_value=(bars[["open","high","low","close","volume"]] <= 0).any(axis=1) #  at least one value in this row True
negative_value.sum()

np.int64(853)

In [33]:
mask = negative_value
all_rejects.append(make_rejects(bars, mask, "INVALID_NUMERIC_RANGE", "bars_1m.parquet"))
bars_cleaned = bars_cleaned.loc[~mask]

#### Invalid OHLC

In [34]:
# OHLC consistency (high must be highest, low must be lowest)
bad_ohlc = (
    (bars.high < bars.low) |
    (bars.high < bars[["open","close"]].max(axis=1)) |
    (bars.low  > bars[["open","close"]].min(axis=1))
)
bad_ohlc.sum()

np.int64(1282)

In [35]:
# Invalid OHLC
mask = bad_ohlc
all_rejects.append(make_rejects(bars, mask, "INVALID_OHLC", "bars_1m.parquet"))
bars_cleaned = bars_cleaned.loc[~mask]


#### Duplicates

In [36]:
# duplicated records
bars.duplicated(subset=["run_id","ticker","ts"]).sum()

np.int64(15120)

In [37]:
# Duplicates (keep the earliest received_at)
bars = bars.sort_values("received_at")
mask = bars.duplicated(["run_id", "ticker", "ts"], keep="first")
all_rejects.append(make_rejects(bars, mask, "DUPLICATE_BAR", "bars_1m.parquet"))
bars_cleaned = bars_cleaned.loc[~mask]

In [38]:
bars["run_id"].value_counts()

run_id
RUN-2025A    1431358
RUN-2025B    1424991
RUN-2025C    1419566
RUN-TEST         260
Name: count, dtype: int64

### Bars with an unknown run

**Finding:** The bars table contains 260 rows with `run_id = RUN-TEST`. This ID does not appear in the flattened `sim_runs` table.

**How it was detected:** I compared the distinct `run_id` values in `bars` with those in `sim_runs_flat`. The `RUN-TEST` bars did not match a known simulation run.

**Decision:** These bars are candidates for rejection because their run cannot be verified. For each rejected bar, the rejects table preserves the raw bar payload, source file (`bars_1m.parquet`), source row, and one rejection reason: `UNKNOWN_RUN`.

In [39]:
mask = bars["run_id"] == "RUN-TEST"
all_rejects.append(make_rejects(bars, mask, "UNKNOWN_RUN", "bars_1m.parquet"))
bars_cleaned = bars_cleaned.loc[~mask]

#### Fake Nulls

In [40]:
FAKE_NULLS = {"", "null", "none", "nan", "na", "n/a", "-", "?"}

for col in ["run_id", "ticker", "ts"]:
    s = bars[col].astype("string").str.strip().str.lower()
    print(col, s.isin(FAKE_NULLS).sum())

run_id 0
ticker 0
ts 0


#### Timestamp Check

In [41]:
bars["ts"].str.len().value_counts()

ts
19    2851052
20    1424849
16        159
10        115
Name: count, dtype: int64

In [42]:
bars.assign(ts_len=bars["ts"].str.len()).groupby("ts_len", dropna=False).head(3).sort_values("ts_len")

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,ts_len
578109,RUN-2025A,MARB,not-a-time,145.35,145.42,145.23,145.26,3640,2025-01-02 15:46:01+00:00,578109,10
1377917,RUN-2025A,INDEX,not-a-time,988.70,988.75,988.34,988.54,129105,2025-01-02 16:31:02+00:00,1377917,10
1911069,RUN-2025B,KELT,not-a-time,88.05,88.10,87.98,88.01,3728,2025-01-02 17:07:01+00:00,1911069,10
1767283,RUN-2025B,HALS,31/03/2025 10:15,58.70,58.73,58.69,58.71,4639,2025-01-02 18:47:01+00:00,1767283,16
2534653,RUN-2025B,XERN,31/03/2025 10:15,17.78,17.78,17.77,17.77,3824,2025-01-02 19:03:02+00:00,2534653,16
2662373,RUN-2025B,ASTR,31/03/2025 10:15,162.12,162.14,162.02,162.02,5536,2025-01-03 16:12:02+00:00,2662373,16
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0,19
578034,RUN-2025A,MARB,2025-01-02 09:30:00,146.30,146.42,146.29,146.38,4849,2025-01-02 14:31:00+00:00,578034,19
3047740,RUN-2025C,ERTL,2025-01-02 09:30:00,117.22,117.81,117.10,117.46,7116,2025-01-02 14:31:00+00:00,3047740,19
2246627,RUN-2025B,ROSS,2025-01-02T14:30:00Z,35.87,35.88,35.85,35.87,896,2025-01-02 14:31:00+00:00,2246627,20


The timestamps `ts` use multiple date formats, standardize them to match the `exchange_calendar`

In [43]:
exchange_calendar.timezone.unique()

<ArrowStringArray>
['America/New_York']
Length: 1, dtype: str

In [44]:
TZ = "America/New_York"    # based on exchange_calendar.csv (timezone column)

ts = bars["ts"].astype("string").str.strip()

# Format A: 2025-01-02 09:30:00  (already local)
fmt_local = pd.to_datetime(ts, format="%Y-%m-%d %H:%M:%S", errors="coerce")

# Format B: 2025-01-02T14:30:00Z  (UTC → convert to local)
fmt_utc = pd.to_datetime(ts, format="%Y-%m-%dT%H:%M:%SZ", errors="coerce", utc=True)
fmt_utc = fmt_utc.dt.tz_convert(TZ).dt.tz_localize(None)

# Format C: 31/03/2025 10:15  (day/month/year) 
fmt_dmy = pd.to_datetime(ts, format="%d/%m/%Y %H:%M", errors="coerce")

# Combine: take the first format that worked
bars_cleaned["ts_local"] = fmt_local.fillna(fmt_utc).fillna(fmt_dmy)

# Remember what was converted (quality flag)
bars["ts_local"] = fmt_local.fillna(fmt_utc).fillna(fmt_dmy)
bars["ts_flag"] = None
bars.loc[fmt_utc.notna(), "ts_flag"] = "TS_CONVERTED_FROM_UTC"
bars.loc[fmt_dmy.notna(), "ts_flag"] = "TS_REFORMATTED_DMY"

In [45]:
# Fixed dates
display(bars_cleaned.loc[bars_cleaned["ts_local"].isna(), "ts"].value_counts())

ts
2025-02-30 10:00:00    79
not-a-time             63
Name: count, dtype: int64

In [46]:
mask = bars["ts_local"].isna()
all_rejects.append(make_rejects(bars, mask, "MALFORMED_TIMESTAMP", "bars_1m.parquet"))
bars_cleaned = bars_cleaned.loc[~mask]

In [47]:
# 2. compare with received_at: lateness should be small and positive for every format

bars_cleaned["end_utc"] = (bars_cleaned["ts_local"] + pd.Timedelta(minutes=1)) \
            .dt.tz_localize(TZ, ambiguous="NaT", nonexistent="NaT") \
            .dt.tz_convert("UTC")                        # 2. safe around clock changes

bars_cleaned["lateness_s"] = (bars_cleaned["received_at"] - bars_cleaned["end_utc"]).dt.total_seconds()

ts_len = bars_cleaned["ts"].astype("string").str.strip().str.len()   # 3. computed here, so it always exists
display(bars_cleaned["lateness_s"].groupby(ts_len).median())

ts
16   -3099238.0
19          1.0
20          1.0
Name: lateness_s, dtype: float64

In [48]:
# Placeholder 
g = bars_cleaned[ts_len == 16]
g["ts"].value_counts().head(10)

ts
31/03/2025 10:15    76
Name: count, dtype: int64

In [49]:
g

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,ts_local,end_utc,lateness_s
9247,RUN-2025A,ARVT,31/03/2025 10:15,134.22,134.33,133.73,133.83,10727,2025-02-04 19:05:01+00:00,9247,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,-4734659.0
152057,RUN-2025A,DALN,31/03/2025 10:15,181.95,183.10,181.94,182.50,22583,2025-01-29 16:29:01+00:00,152057,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,-5262419.0
231732,RUN-2025A,ERTL,31/03/2025 10:15,78.05,78.06,77.98,78.03,3177,2025-05-27 17:07:01+00:00,231732,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,4935061.0
275995,RUN-2025A,FENM,31/03/2025 10:15,63.15,63.16,63.12,63.12,1781,2025-05-12 17:00:00+00:00,275995,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,3638640.0
291246,RUN-2025A,GALR,31/03/2025 10:15,20.61,20.63,20.57,20.58,2780,2025-01-09 18:45:01+00:00,291246,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,-6982259.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4041977,RUN-2025C,ZENM,31/03/2025 10:15,50.93,51.10,50.65,50.87,941,2025-01-29 19:23:02+00:00,4041977,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,-5251978.0
4108862,RUN-2025C,ASTR,31/03/2025 10:15,218.35,218.35,218.27,218.34,2161,2025-04-09 18:21:00+00:00,4108862,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,792300.0
4137627,RUN-2025C,BRIS,31/03/2025 10:15,81.96,83.02,81.89,82.66,2294,2025-01-29 20:54:02+00:00,4137627,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,-5246518.0
4196757,RUN-2025C,CELK,31/03/2025 10:15,132.00,132.02,131.73,131.81,1654,2025-05-09 17:06:01+00:00,4196757,2025-03-31 10:15:00,2025-03-31 14:16:00+00:00,3379801.0


In [50]:
placeholder = bars_cleaned["ts"].astype("string").str.strip() == "31/03/2025 10:15"
mask = placeholder
all_rejects.append(make_rejects(bars_cleaned, mask, "MALFORMED_TIMESTAMP", "bars_1m.parquet"))
bars_cleaned = bars_cleaned.loc[~mask]
print(mask.sum())    # 159

76


In [51]:
## Adding columns
bars_cleaned["trading_date"] = bars_cleaned["ts_local"].dt.date
bars_cleaned["bar_time"]     = bars_cleaned["ts_local"].dt.time

In [52]:
bars_cleaned

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,ts_local,end_utc,lateness_s,trading_date,bar_time
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0,2025-01-02 09:30:00,2025-01-02 14:31:00+00:00,0.0,2025-01-02,09:30:00
1,RUN-2025A,ARVT,2025-01-02 09:31:00,119.93,119.98,119.35,119.39,45001,2025-01-02 14:32:01+00:00,1,2025-01-02 09:31:00,2025-01-02 14:32:00+00:00,1.0,2025-01-02,09:31:00
2,RUN-2025A,ARVT,2025-01-02 09:32:00,119.39,119.66,119.30,119.53,5825,2025-01-02 14:33:00+00:00,2,2025-01-02 09:32:00,2025-01-02 14:33:00+00:00,0.0,2025-01-02,09:32:00
3,RUN-2025A,ARVT,2025-01-02 09:33:00,119.53,119.57,119.23,119.35,8195,2025-01-02 14:34:00+00:00,3,2025-01-02 09:33:00,2025-01-02 14:34:00+00:00,0.0,2025-01-02,09:33:00
4,RUN-2025A,ARVT,2025-01-02 09:34:00,119.35,119.44,119.33,119.38,5631,2025-01-02 14:35:01+00:00,4,2025-01-02 09:34:00,2025-01-02 14:35:00+00:00,1.0,2025-01-02,09:34:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4271219,RUN-2025C,INDEX,2025-06-30 15:55:00,1234.18,1234.59,1234.17,1234.43,126336,2025-06-30 19:56:00+00:00,4271219,2025-06-30 15:55:00,2025-06-30 19:56:00+00:00,0.0,2025-06-30,15:55:00
4271220,RUN-2025C,INDEX,2025-06-30 15:56:00,1234.43,1235.04,1234.19,1235.00,135129,2025-06-30 19:57:02+00:00,4271220,2025-06-30 15:56:00,2025-06-30 19:57:00+00:00,2.0,2025-06-30,15:56:00
4271221,RUN-2025C,INDEX,2025-06-30 15:57:00,1235.00,1235.18,1234.12,1234.39,126898,2025-06-30 19:58:00+00:00,4271221,2025-06-30 15:57:00,2025-06-30 19:58:00+00:00,0.0,2025-06-30,15:57:00
4271222,RUN-2025C,INDEX,2025-06-30 15:58:00,1234.39,1234.42,1234.13,1234.37,115500,2025-06-30 19:59:02+00:00,4271222,2025-06-30 15:58:00,2025-06-30 19:59:00+00:00,2.0,2025-06-30,15:58:00


#### Cleansing Results

In [53]:
# Results
rejects = pd.concat(all_rejects, ignore_index=True)
clean = df

print(rejects["reason"].value_counts())

reason
DUPLICATE_BAR             15120
INVALID_OHLC               1282
MISSING_REQUIRED_FIELD      854
INVALID_NUMERIC_RANGE       853
MALFORMED_TIMESTAMP         342
UNKNOWN_RUN                 260
Name: count, dtype: int64


### 4. EDA `exchange_calendar.csv`

#### 4.1 Review

In [54]:
calendar = pd.read_csv(DATA_DIR / "exchange_calendar.csv", dtype="string",
                       keep_default_na=False, na_values=[""])
calendar.info()
calendar.head()

<class 'pandas.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   date            129 non-null    string
 1   is_trading_day  129 non-null    string
 2   session_open    124 non-null    string
 3   session_close   124 non-null    string
 4   timezone        129 non-null    string
 5   note            6 non-null      string
dtypes: string(6)
memory usage: 11.3 KB


,date,is_trading_day,session_open,session_close,timezone,note
0,2025-01-02,true,09:30,16:00,America/New_York,<NA>
1,2025-01-03,true,09:30,16:00,America/New_York,<NA>
2,2025-01-06,true,09:30,16:00,America/New_York,<NA>
3,2025-01-07,true,09:30,16:00,America/New_York,<NA>
4,2025-01-08,true,09:30,16:00,America/New_York,<NA>


In [55]:
# wrong data types. Everything was read as text on purpose; what does each column hold?
print("is_trading_day values:", calendar["is_trading_day"].unique().tolist())
print("timezone values:", calendar["timezone"].unique().tolist())
calendar["date"] = pd.to_datetime(calendar["date"], errors="coerce")
calendar["is_trading_day"] = calendar["is_trading_day"].map({"true": True, "false": False})
print("dates that failed to parse:", calendar["date"].isna().sum())
print("is_trading_day values that failed to map:", calendar["is_trading_day"].isna().sum())

is_trading_day values: ['true', 'false']
timezone values: ['America/New_York']
dates that failed to parse: 0
is_trading_day values that failed to map: 0


#### Finding 4.1 Booleans and dates are text

data-quality issues:
1. is_trading_day was stored as text instead of Boolean values. 
2. date was stored as text instead of datetime values.

- Both convert cleanly: 
    - no date failed `to_datetime(errors="coerce")`
    - no boolean failed to map
- Decision: Convert columns explicitly to their appropriate data types and check for conversion failures.

#### 4.2 Keys

In [56]:
print("date unique?", calendar["date"].is_unique)
print("range:", calendar["date"].min().date(), "to", calendar["date"].max().date())
# Lab 3, section 4.4: derive features from dates. No weekend should appear; no weekday should be missing.
calendar["day_name"] = calendar["date"].dt.day_name()
print(calendar["day_name"].value_counts())
all_weekdays = pd.bdate_range(calendar["date"].min(), calendar["date"].max())
print("weekdays missing from the calendar:", len(set(all_weekdays) - set(calendar["date"])))

date unique? True
range: 2025-01-02 to 2025-07-01
day_name
Thursday     26
Friday       26
Monday       26
Tuesday      26
Wednesday    25
Name: count, dtype: int64
weekdays missing from the calendar: 0


#### Finding 4.2 Coverage is complete
- dataset contains every expected trading-calendar date within the period (129 unique dates from 2025-01-02 to 2025-07-01), without gaps or unexpected dates.
-  no weekends as stock exchanges don't trade on weekends
- no missing weekdays
- one time zone (`America/New_York`), which makes date and time comparisons more reliable
- Decision: Use it as delivered as it is complete

#### 4.3 Closed days and session hours

In [57]:
def missing_table(df):
    # how many values are missing, and what percentage
    return (pd.DataFrame({"missing": df.isnull().sum(),
                          "missing_%": (df.isnull().mean() * 100).round(3)})
            .sort_values("missing", ascending=False))

missing_table(calendar)

,missing,missing_%
note,123,95.349
session_open,5,3.876
session_close,5,3.876
date,0,0.000
is_trading_day,0,0.000
timezone,0,0.000
day_name,0,0.000


In [58]:
# the missing session hours should be the same as the closed days (structurally missing)
calendar["session_open"].isnull().groupby(calendar["is_trading_day"]).mean()

is_trading_day
False    1.0
True     0.0
Name: session_open, dtype: float64

In [59]:
calendar[~calendar["is_trading_day"] | calendar["note"].notna()]

,date,is_trading_day,session_open,session_close,timezone,note,day_name
12,2025-01-20,False,<NA>,<NA>,America/New_York,Exchange holiday,Monday
32,2025-02-17,False,<NA>,<NA>,America/New_York,Presidents' Day,Monday
75,2025-04-17,True,09:30,13:00,America/New_York,early close,Thursday
76,2025-04-18,False,<NA>,<NA>,America/New_York,Good Friday,Friday
102,2025-05-26,False,<NA>,<NA>,America/New_York,Memorial Day,Monday
120,2025-06-19,False,<NA>,<NA>,America/New_York,Juneteenth,Thursday


#### Finding 4.3 Five holidays (US Market Holidays), and their missing session hours are structural

- The 5 missing `session_open` / `session_close` values are exactly the 5 closed days (missing rate 100% when closed, 0% when open): 01-20, 02-17, 04-18, 05-26, 06-19.
- These are not lost data. Any bar dated on one of these days should not exist.
- Decision: 
    - Keep the nulls & don't fill these missing values. 
    - If a price bar appears on one of these five holidays, it should be classified as: `OUTSIDE_SESSION`

#### 4.4 trading-session lengths

In [60]:
#  defining the batch window to compare dates
WIN_START, WIN_END = pd.Timestamp("2025-01-02"), pd.Timestamp("2025-06-30")

# keep only trading dates
trading = calendar[calendar["is_trading_day"]].copy()

# calculate session duration, how many minutes the stock exchange operates each day. opening time - closing time in minutes
trading["session_minutes"] = ((pd.to_datetime(trading["session_close"], format="%H:%M")
                               - pd.to_datetime(trading["session_open"], format="%H:%M"))
                              .dt.total_seconds() // 60).astype(int)

# Count trading days in the batch window
print("trading days in the batch window:", trading["date"].between(WIN_START, WIN_END).sum())

# Count session durations
trading["session_minutes"].value_counts()

trading days in the batch window: 123


session_minutes
390    123
210      1
Name: count, dtype: int64

#### Finding 3.4 One trading day has early close: 2025-04-17 ends at 13:00

- 123 trading days in the batch window. 122 have 390 minutes (09:30–16:00); 04-17 has 210 minutes (09:30–13:00).
- Bars between 13:00 and 16:00 on 04-17 are outside the session, and the expected-bars count for that day is 210, not 390.
- Decision: Read session length from the calendar per day and do not assume 390

#### 4.5 Daylight saving

In [61]:
# exchange clock is America/New_York. What is the 09:30 open in UTC, day by day?
open_utc = (trading["date"] + pd.Timedelta(hours=9, minutes=30)).dt.tz_localize("America/New_York") \
               .dt.tz_convert("UTC").dt.strftime("%H:%M")
open_utc.groupby(trading["date"].dt.to_period("M")).value_counts().unstack(fill_value=0)

date,13:30,14:30
date,,
2025-01,0,21
2025-02,0,19
2025-03,16,5
2025-04,21,0
2025-05,21,0
2025-06,20,0
2025-07,1,0


#### Finding 4.5 Daylight saving time starts on 2025-03-09

- Opening time changes during march: The 09:30 New York open is 14:30 UTC until 03-07 and 13:30 UTC from 03-10.
- A source that stores UTC shows a one-hour jump in its opening time in March
- Decision: Convert time zones with a real time-zone library (`tz_localize` / `tz_convert`), not with a fixed offset

### EDA Events

In [62]:
events_clean = events_flat.copy()

In [63]:
events_clean.head()

,_source_row,event_id,type,time,security_ticker,details_session,details_duration_minutes,details_ratio
0,1,EV0001,earnings,2025-01-10T09:30:00-05:00,DALN,pre_open,NaN,NaN
1,2,EV0002,earnings,2025-01-14T09:30:00-05:00,BRIS,pre_open,NaN,NaN
2,3,EV0003,earnings,2025-01-15T09:30:00-05:00,SOLK,pre_open,NaN,NaN
3,4,EV0004,earnings,2025-01-24T09:30:00-05:00,XERN,pre_open,NaN,NaN
4,5,EV0005,earnings,2025-01-30T09:30:00-05:00,VALX,pre_open,NaN,NaN


In [64]:
events_clean.info()



<class 'pandas.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   _source_row               62 non-null     int64  
 1   event_id                  62 non-null     str    
 2   type                      62 non-null     str    
 3   time                      62 non-null     str    
 4   security_ticker           62 non-null     str    
 5   details_session           60 non-null     str    
 6   details_duration_minutes  1 non-null      float64
 7   details_ratio             1 non-null      str    
dtypes: float64(1), int64(1), str(6)
memory usage: 7.1 KB


In [65]:
print("Duplicate event_id:", events_clean["event_id"].duplicated().sum())
print("Same event, different ID:",events_clean.duplicated(["type", "security_ticker", "time"]).sum())


Duplicate event_id: 0
Same event, different ID: 0


#### Dates fixing

In [66]:
# changes the text in time into real date-times on the UTC
events_clean["time_utc"] = pd.to_datetime(events_flat["time"], utc=True, errors="coerce")

In [67]:
# Add local time so change from UTC to Newyork
events_clean["time_local"] = events_clean["time_utc"].dt.tz_convert("America/New_York").dt.tz_localize(None)

In [68]:
events_clean["time_utc"].isna().sum()

np.int64(0)

In [69]:
events_clean["time_local"].min(), events_clean["time_local"].max()

(Timestamp('2025-01-10 09:30:00'), Timestamp('2025-06-18 09:30:00'))

In [70]:
events_clean

,_source_row,event_id,type,time,security_ticker,details_session,details_duration_minutes,details_ratio,time_utc,time_local
0,1,EV0001,earnings,2025-01-10T09:30:00-05:00,DALN,pre_open,NaN,NaN,2025-01-10 14:30:00+00:00,2025-01-10 09:30:00
1,2,EV0002,earnings,2025-01-14T09:30:00-05:00,BRIS,pre_open,NaN,NaN,2025-01-14 14:30:00+00:00,2025-01-14 09:30:00
2,3,EV0003,earnings,2025-01-15T09:30:00-05:00,SOLK,pre_open,NaN,NaN,2025-01-15 14:30:00+00:00,2025-01-15 09:30:00
3,4,EV0004,earnings,2025-01-24T09:30:00-05:00,XERN,pre_open,NaN,NaN,2025-01-24 14:30:00+00:00,2025-01-24 09:30:00
4,5,EV0005,earnings,2025-01-30T09:30:00-05:00,VALX,pre_open,NaN,NaN,2025-01-30 14:30:00+00:00,2025-01-30 09:30:00
...,...,...,...,...,...,...,...,...,...,...
57,58,EV0058,earnings,2025-06-09T09:30:00-04:00,UMBT,pre_open,NaN,NaN,2025-06-09 13:30:00+00:00,2025-06-09 09:30:00
58,59,EV0059,earnings,2025-06-11T09:30:00-04:00,ARVT,pre_open,NaN,NaN,2025-06-11 13:30:00+00:00,2025-06-11 09:30:00
59,60,EV0060,earnings,2025-06-12T09:30:00-04:00,ORBL,pre_open,NaN,NaN,2025-06-12 13:30:00+00:00,2025-06-12 09:30:00
60,61,EV0061,earnings,2025-06-13T09:30:00-04:00,ASTR,pre_open,NaN,NaN,2025-06-13 13:30:00+00:00,2025-06-13 09:30:00


#### Understanding missing values

In [71]:
events_clean.isna().sum()

_source_row                  0
event_id                     0
type                         0
time                         0
security_ticker              0
details_session              2
details_duration_minutes    61
details_ratio               61
time_utc                     0
time_local                   0
dtype: int64

In [72]:
events_clean["type"].value_counts()

type
earnings    60
halt         1
split        1
Name: count, dtype: int64

- details_session, for earnings. Earnings is when a company announces its profits
- details_ratio, for splits. A split is when a company cuts each share into smaller pieces. The total value stays the same
- details_duration_minutes, for halts. A halt is when the exchange pauses trading in one stock for a while, often because of big pending news

Split ratio stored as text. details_ratio is a string in new:old form ("2:1")
You divide the old prices by 2 because each old share became 2 new shares. One old share at $100 equals two new shares at $50 each. Dividing by 2 puts the old prices in "new share" terms

In [73]:
events_clean[events_clean["type"] == "split"]["details_ratio"].describe()        # ratios > 0?

count       1
unique      1
top       2:1
freq        1
Name: details_ratio, dtype: object

In [74]:
events_clean[events_clean["type"] == "split"]["details_ratio"].unique()

<ArrowStringArray>
['2:1']
Length: 1, dtype: str

In [75]:
events_clean[events_clean["type"] == "halt"]["details_duration_minutes"].describe()  # durations > 0?


count     1.0
mean     45.0
std       NaN
min      45.0
25%      45.0
50%      45.0
75%      45.0
max      45.0
Name: details_duration_minutes, dtype: float64

Cleaning the columns event_id and security_ticker to remove any spaces or lower cases

In [76]:
events_clean  = clean_text_values_check(events_clean,  columns=["event_id", "security_ticker"])

1 event uses a pre-rename ticker. EV for ORBT on 2025-03-12 doesn't match symbols.csv (which only lists the current ticker ORBX). symbol_history.csv shows ORBT → ORBX effective 2025-03-17, so on the event date ORBT was valid. Resolved to SEC017; not rejected.

In [77]:
# unknown tickers in comparison with symbols
events_clean["security_ticker"].isin(symbols_clean["ticker"]).value_counts()     

security_ticker
True     61
False     1
Name: count, dtype: int64

In [78]:
unknown = ~events_clean["security_ticker"].isin(symbols_clean["ticker"])
events_clean[unknown]

,_source_row,event_id,type,time,security_ticker,details_session,details_duration_minutes,details_ratio,time_utc,time_local
22,23,EV0023,earnings,2025-03-12T09:30:00-04:00,ORBT,pre_open,NaN,NaN,2025-03-12 13:30:00+00:00,2025-03-12 09:30:00


In [79]:
symbol_history[symbol_history["change_type"] == "ticker"] 

,security_id,change_type,old_value,new_value,effective_date,_source_row
0,SEC017,ticker,ORBT,ORBX,2025-03-17,0


In [80]:
symbols_clean[symbols_clean["ticker"] == "ORBX"]

,security_id,ticker,company_name,sector,listing_date,delisting_date,shares_outstanding,_source_row
16,SEC017,ORBX,QUAON INDUSTRIES,INDUSTRIALS,2020-05-03,NaN,725000000,16


In [81]:
symbol_history[symbol_history["security_id"] == "SEC017"]


,security_id,change_type,old_value,new_value,effective_date,_source_row
0,SEC017,ticker,ORBT,ORBX,2025-03-17,0


### Mergeing Datasets:

### Merge `sim_runs` with `bar History` 

RUN-2025B appears twice in sim_runs; same config_version, but created_at differs. 
For the bar join, should retain one run record and exclude conflicting "created_at" as the relation is many_to_one

In [82]:
sim_runs_clean = (
    sim_runs_flat[["run_id", "config_version"]]
    .drop_duplicates(subset=["run_id", "config_version"])
)

In [83]:
bars_with_runs= bars_cleaned.merge(
    sim_runs_clean,
    on="run_id",
    how="left",
    validate="many_to_one"
)

In [84]:
print("bars before clean :", len(bars))
print("bars after clean:", len(bars_cleaned))
print("bars after merge: ", len(bars_with_runs))
print("unmatched bars:", bars_with_runs["config_version"].isna().sum())

bars before clean : 4276175
bars after clean: 4257598
bars after merge:  4257598
unmatched bars: 0


In [85]:
bars_with_runs.loc[
    bars_with_runs["config_version"].isna(),
    "run_id"
].drop_duplicates()

Series([], Name: run_id, dtype: str)

### Merge `Symbols` with `Symbols History` 

From the EDA of both Datasets we noticed that the link is the Security_id, and Symbols History includes only 3 entries for 3 companies in which there is a change that happened, so we will merge them to the data in Symbols dataset.


In [86]:
symbols_clean.head()

,security_id,ticker,company_name,sector,listing_date,delisting_date,shares_outstanding,_source_row
0,SEC001,ARVT,ARVON SYSTEMS,TECHNOLOGY,2019-01-03,NaN,772000000,0
1,SEC002,BELX,BELIA CAPITAL,FINANCIALS,2020-02-03,NaN,193000000,1
2,SEC003,CORB,CORON THERAPEUTICS,HEALTHCARE,2021-03-03,NaN,660000000,2
3,SEC004,DALN,DALIA ENERGY,ENERGY,2022-04-03,NaN,192000000,3
4,SEC005,ERTL,ERTON INDUSTRIES,INDUSTRIALS,2023-05-03,NaN,141000000,4


In [87]:
History_cleaning.head(3)

,security_id,change_type,old_value,new_value,effective_date,_source_row
0,SEC017,TICKER,ORBT,ORBX,2025-03-17,0
1,SEC022,SECTOR,ENERGY,TECHNOLOGY,2025-05-01,1
2,SEC010,SHARES_OUTSTANDING,207000000,414000000,2025-04-01,2


First we need to convert History_Cleaning dataset into wide format to better match the format of symbols_clean columns.

In [88]:
cols = (
    History_cleaning[
        ['security_id', 'effective_date', '_source_row']
    ]
    .drop_duplicates()
)

History_wide = History_cleaning.pivot(
    index='security_id',
    columns='change_type',
    values='new_value'
).reset_index()

History_wide = History_wide.merge(
    cols,
    on='security_id',
    how='left'
)

In [89]:
History_wide.head(3)

,security_id,SECTOR,SHARES_OUTSTANDING,TICKER,effective_date,_source_row
0,SEC010,NaN,414000000,NaN,2025-04-01,2
1,SEC017,NaN,NaN,ORBX,2025-03-17,0
2,SEC022,TECHNOLOGY,NaN,NaN,2025-05-01,1


Now we will need to match the Column names and types with Symbols_clean dataset.

In [90]:
History_wide.columns = History_wide.columns.str.lower()

In [91]:
print(symbols_clean.dtypes)
print(History_wide.dtypes)

security_id             str
ticker                  str
company_name            str
sector                  str
listing_date            str
delisting_date          str
shares_outstanding    int64
_source_row           int64
dtype: object
security_id             str
sector                  str
shares_outstanding      str
ticker                  str
effective_date          str
_source_row           int64
dtype: object


In [92]:
History_wide= History_wide.rename(columns={'effective_date':'listing_date'})
History_wide['shares_outstanding'] = History_wide['shares_outstanding'].astype('Int64')

First we matched the names of the "effective_date" which indicates "the first trading day the new value applies", and "listing_date" which indicates "first trading day" so they can be both under the same name "listing_date".
Secondly we can see that there is difference in the type of "Share_outstanding" it is integr in Symbols_clean and string in the History_wide, so we need to matched it to be Integr as well. 

In [93]:
History_wide.head(3)

,security_id,sector,shares_outstanding,ticker,listing_date,_source_row
0,SEC010,NaN,414000000,NaN,2025-04-01,2
1,SEC017,NaN,<NA>,ORBX,2025-03-17,0
2,SEC022,TECHNOLOGY,<NA>,NaN,2025-05-01,1


In [94]:
common_cols = list(
    symbols_clean.columns.intersection(History_wide.columns)
)

Symbols_History_merged = symbols_clean.merge(
    History_wide,
    on=common_cols,
    how='outer'
)

In [95]:
Symbols_History_merged.head()

,security_id,ticker,company_name,sector,listing_date,delisting_date,shares_outstanding,_source_row
0,SEC001,ARVT,ARVON SYSTEMS,TECHNOLOGY,2019-01-03,NaN,772000000,0
1,SEC002,BELX,BELIA CAPITAL,FINANCIALS,2020-02-03,NaN,193000000,1
2,SEC003,CORB,CORON THERAPEUTICS,HEALTHCARE,2021-03-03,NaN,660000000,2
3,SEC004,DALN,DALIA ENERGY,ENERGY,2022-04-03,NaN,192000000,3
4,SEC005,ERTL,ERTON INDUSTRIES,INDUSTRIALS,2023-05-03,NaN,141000000,4


In [96]:
Symbols_History_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   security_id         33 non-null     str  
 1   ticker              31 non-null     str  
 2   company_name        30 non-null     str  
 3   sector              31 non-null     str  
 4   listing_date        33 non-null     str  
 5   delisting_date      1 non-null      str  
 6   shares_outstanding  31 non-null     Int64
 7   _source_row         33 non-null     int64
dtypes: Int64(1), int64(1), str(6)
memory usage: 3.6 KB


In [97]:
Symbols_History_merged.tail(10)

,security_id,ticker,company_name,sector,listing_date,delisting_date,shares_outstanding,_source_row
23,SEC022,VALX,VALIA ENERGY,ENERGY,2020-10-03,NaN,705000000,21
24,SEC022,NaN,NaN,TECHNOLOGY,2025-05-01,NaN,<NA>,1
25,SEC023,WESB,WESON INDUSTRIES,INDUSTRIALS,2021-11-03,NaN,72000000,22
26,SEC024,XERN,XERIA BRANDS,CONSUMER,2022-12-03,NaN,296000000,23
27,SEC025,YARL,YARON SYSTEMS,TECHNOLOGY,2023-01-03,2025-04-30,842000000,24
28,SEC026,ZENM,ZENIA CAPITAL,FINANCIALS,2019-02-03,NaN,462000000,25
29,SEC027,ASTR,ASTON THERAPEUTICS,HEALTHCARE,2020-03-03,NaN,250000000,26
30,SEC028,BRIS,BRIIA ENERGY,ENERGY,2021-04-03,NaN,717000000,27
31,SEC029,CELK,CELON INDUSTRIES,INDUSTRIALS,2025-03-03,NaN,670000000,28
32,SEC030,DOVV,DOVIA BRANDS,CONSUMER,2025-05-12,NaN,740000000,29


We can see that there are some missing values in the merged rows, and since it is of a same company we will match their ticker,company_name and sector.

In [98]:
cols_to_fill = ['ticker', 'company_name', 'sector']

Symbols_History_merged[cols_to_fill] = (
    Symbols_History_merged
    .groupby('security_id')[cols_to_fill]
    .transform(lambda x: x.ffill().bfill())
)

In [99]:
Symbols_History_merged.isna().sum()

security_id            0
ticker                 0
company_name           0
sector                 0
listing_date           0
delisting_date        32
shares_outstanding     2
_source_row            0
dtype: int64

The merge between Symbols and Symbols_History is done, but since the data in the Symbols_History dataset captured a change in the company, we can't merge the shares_outstanding for the before and after the change.

In [100]:
symbols_clean

,security_id,ticker,company_name,sector,listing_date,delisting_date,shares_outstanding,_source_row
0,SEC001,ARVT,ARVON SYSTEMS,TECHNOLOGY,2019-01-03,NaN,772000000,0
1,SEC002,BELX,BELIA CAPITAL,FINANCIALS,2020-02-03,NaN,193000000,1
2,SEC003,CORB,CORON THERAPEUTICS,HEALTHCARE,2021-03-03,NaN,660000000,2
3,SEC004,DALN,DALIA ENERGY,ENERGY,2022-04-03,NaN,192000000,3
4,SEC005,ERTL,ERTON INDUSTRIES,INDUSTRIALS,2023-05-03,NaN,141000000,4
5,SEC006,FENM,FENIA BRANDS,CONSUMER,2019-06-03,NaN,483000000,5
6,SEC007,GALR,GALON SYSTEMS,TECHNOLOGY,2020-07-03,NaN,118000000,6
7,SEC008,HALS,HALIA CAPITAL,FINANCIALS,2021-08-03,NaN,410000000,7
8,SEC009,ISTK,ISTON THERAPEUTICS,HEALTHCARE,2022-09-03,NaN,320000000,8
9,SEC010,JORV,JORIA ENERGY,ENERGY,2023-10-03,NaN,414000000,9


In [101]:
Symbols_History_merged[Symbols_History_merged["security_id"] == "SEC017"]

,security_id,ticker,company_name,sector,listing_date,delisting_date,shares_outstanding,_source_row
17,SEC017,ORBX,QUAON INDUSTRIES,INDUSTRIALS,2020-05-03,NaN,725000000,16
18,SEC017,ORBX,QUAON INDUSTRIES,INDUSTRIALS,2025-03-17,NaN,<NA>,0


### Trial 2

In [102]:
History_wide = History_cleaning.pivot(
    index='security_id',
    columns='change_type',
    values='old_value'
).reset_index()

History_wide2 = History_wide.merge(
    cols,
    on='security_id',
    how='left'
)

In [103]:
Symbols_History_merged_V2 = symbols_clean.copy()
Symbols_History_merged_V2["valid_from"] = pd.to_datetime(Symbols_History_merged_V2["listing_date"])
Symbols_History_merged_V2["valid_to"] = pd.to_datetime(Symbols_History_merged_V2["delisting_date"]).fillna(pd.Timestamp.max)

for _, h in History_wide2.iterrows():
    date = pd.to_datetime(h["effective_date"])
    company = Symbols_History_merged_V2["security_id"] == h["security_id"]

    before = Symbols_History_merged_V2[company].copy()
    if pd.notna(h["TICKER"]):             before["ticker"] = h["TICKER"]
    if pd.notna(h["SECTOR"]):             before["sector"] = h["SECTOR"]
    if pd.notna(h["SHARES_OUTSTANDING"]): before["shares_outstanding"] = int(h["SHARES_OUTSTANDING"])
    before["valid_to"] = date

    Symbols_History_merged_V2.loc[company, "valid_from"] = date
    Symbols_History_merged_V2 = pd.concat([Symbols_History_merged_V2, before], ignore_index=True)

Symbols_History_merged_V2 = Symbols_History_merged_V2.sort_values(["security_id", "valid_from"]).reset_index(drop=True)

In [104]:
Symbols_History_merged_V2[Symbols_History_merged_V2["security_id"].isin(["SEC017", "SEC010"])]

,security_id,ticker,company_name,sector,listing_date,delisting_date,shares_outstanding,_source_row,valid_from,valid_to
9,SEC010,JORV,JORIA ENERGY,ENERGY,2023-10-03,NaN,207000000,9,2023-10-03,2025-04-01 00:00:00.000000000
10,SEC010,JORV,JORIA ENERGY,ENERGY,2023-10-03,NaN,414000000,9,2025-04-01,2262-04-11 23:47:16.854775807
17,SEC017,ORBT,QUAON INDUSTRIES,INDUSTRIALS,2020-05-03,NaN,725000000,16,2020-05-03,2025-03-17 00:00:00.000000000
18,SEC017,ORBX,QUAON INDUSTRIES,INDUSTRIALS,2020-05-03,NaN,725000000,16,2025-03-17,2262-04-11 23:47:16.854775807


### Merge `events` with `Symbols` & `Symbols History` 

In [105]:
m = events_clean.merge(
    Symbols_History_merged_V2, left_on="security_ticker", right_on="ticker", how="left")

# keep rows where the event fits the period, OR where there's no event at all
valid = ((m["time_local"] >= m["valid_from"]) & (m["time_local"] < m["valid_to"])) \
        | m["event_id"].isna()

events_with_symbols = m[valid]
print(len(events_clean), len(events_with_symbols))   

62 59


In [109]:
lost = events_clean[~events_clean["event_id"].isin(events_with_symbols["event_id"])]
lost[["event_id", "type", "security_ticker", "time_local"]]

,event_id,type,security_ticker,time_local
15,EV0016,earnings,CELK,2025-02-14 09:30:00
20,EV0021,earnings,DOVV,2025-03-06 09:30:00
54,EV0055,earnings,YARL,2025-05-30 09:30:00


A company can't hold an earnings announcement on an exchange before it's listed there, or after it's left. So on those dates, these tickers matched no company valid on the date.

In [110]:
Symbols_History_merged_V2[Symbols_History_merged_V2["ticker"].isin(["CELK", "DOVV","YARL"])]

,security_id,ticker,company_name,sector,listing_date,delisting_date,shares_outstanding,_source_row,valid_from,valid_to
27,SEC025,YARL,YARON SYSTEMS,TECHNOLOGY,2023-01-03,2025-04-30,842000000,24,2023-01-03,2025-04-30 00:00:00.000000000
31,SEC029,CELK,CELON INDUSTRIES,INDUSTRIALS,2025-03-03,NaN,670000000,28,2025-03-03,2262-04-11 23:47:16.854775807
32,SEC030,DOVV,DOVIA BRANDS,CONSUMER,2025-05-12,NaN,740000000,29,2025-05-12,2262-04-11 23:47:16.854775807


In [111]:
lost_mask = ~events_clean["event_id"].isin(events_with_symbols["event_id"])
all_rejects.append(make_rejects(events_clean, lost_mask, "UNKNOWN_SYMBOL", "events.json"))

In [112]:
events_with_symbols.head()

,_source_row_x,event_id,type,time,security_ticker,details_session,details_duration_minutes,details_ratio,time_utc,time_local,security_id,ticker,company_name,sector,listing_date,delisting_date,shares_outstanding,_source_row_y,valid_from,valid_to
0,1,EV0001,earnings,2025-01-10T09:30:00-05:00,DALN,pre_open,NaN,NaN,2025-01-10 14:30:00+00:00,2025-01-10 09:30:00,SEC004,DALN,DALIA ENERGY,ENERGY,2022-04-03,NaN,192000000,3,2022-04-03,2262-04-11 23:47:16.854775807
1,2,EV0002,earnings,2025-01-14T09:30:00-05:00,BRIS,pre_open,NaN,NaN,2025-01-14 14:30:00+00:00,2025-01-14 09:30:00,SEC028,BRIS,BRIIA ENERGY,ENERGY,2021-04-03,NaN,717000000,27,2021-04-03,2262-04-11 23:47:16.854775807
2,3,EV0003,earnings,2025-01-15T09:30:00-05:00,SOLK,pre_open,NaN,NaN,2025-01-15 14:30:00+00:00,2025-01-15 09:30:00,SEC019,SOLK,SOLON SYSTEMS,TECHNOLOGY,2022-07-03,NaN,421000000,18,2022-07-03,2262-04-11 23:47:16.854775807
3,4,EV0004,earnings,2025-01-24T09:30:00-05:00,XERN,pre_open,NaN,NaN,2025-01-24 14:30:00+00:00,2025-01-24 09:30:00,SEC024,XERN,XERIA BRANDS,CONSUMER,2022-12-03,NaN,296000000,23,2022-12-03,2262-04-11 23:47:16.854775807
4,5,EV0005,earnings,2025-01-30T09:30:00-05:00,VALX,pre_open,NaN,NaN,2025-01-30 14:30:00+00:00,2025-01-30 09:30:00,SEC022,VALX,VALIA ENERGY,ENERGY,2020-10-03,NaN,705000000,21,2020-10-03,2025-05-01 00:00:00.000000000


In [113]:
events_with_symbols.shape

(59, 20)

### Merge `bars_with_runs` with `exchange_calendar.csv` 

with `is_trading_day`= True and `bars_with_runs`.`trading_date`= `exchange_calendar`.`date` and `bars_with_runs`.`bar_time` is between `exchange_calendar`.`session_open` and `session_close` all as IN_SESSION ELSE OUTSIDE_SESSION

In [ ]:
exchange_calendar = pd.read_csv(DATA_DIR / "exchange_calendar.csv")

In [ ]:
exchange_calendar.head()

,date,is_trading_day,session_open,session_close,timezone,note
0,2025-01-02,True,09:30,16:00,America/New_York,NaN
1,2025-01-03,True,09:30,16:00,America/New_York,NaN
2,2025-01-06,True,09:30,16:00,America/New_York,NaN
3,2025-01-07,True,09:30,16:00,America/New_York,NaN
4,2025-01-08,True,09:30,16:00,America/New_York,NaN


In [ ]:
print(exchange_calendar.dtypes)

date               str
is_trading_day    bool
session_open       str
session_close      str
timezone           str
note               str
dtype: object


In [ ]:
print(bars_with_runs.dtypes)

run_id                            str
ticker                            str
ts                                str
open                          float64
high                          float64
low                           float64
close                         float64
volume                          int64
received_at       datetime64[ms, UTC]
_source_row                     int64
ts_local               datetime64[us]
end_utc           datetime64[us, UTC]
lateness_s                    float64
trading_date                   object
bar_time                       object
config_version                    str
dtype: object


We should convert the `exchange_calendar` date column to datetime.date and `session_open` and `session_close` to datetime.time

In [ ]:
exchange_calendar["date"] = pd.to_datetime(
    exchange_calendar["date"]
).dt.date

In [ ]:
exchange_calendar["session_open"] = pd.to_datetime(
    exchange_calendar["session_open"], format="%H:%M"
).dt.time

In [ ]:
exchange_calendar["session_close"] = pd.to_datetime(
    exchange_calendar["session_close"], format="%H:%M"
).dt.time

In [ ]:
print(type(bars_with_runs["trading_date"].iloc[0]))
print(type(exchange_calendar["date"].iloc[0]))

print(
    repr(bars_with_runs["trading_date"].iloc[0]),
    repr(exchange_calendar["date"].iloc[0])
)

<class 'datetime.date'>
<class 'datetime.date'>
datetime.date(2025, 1, 2) datetime.date(2025, 1, 2)


In [ ]:
print(exchange_calendar.dtypes)

date              object
is_trading_day      bool
session_open      object
session_close     object
timezone             str
note                 str
dtype: object


In [ ]:
print(bars_with_runs.dtypes)

run_id                            str
ticker                            str
ts                                str
open                          float64
high                          float64
low                           float64
close                         float64
volume                          int64
received_at       datetime64[ms, UTC]
_source_row                     int64
ts_local               datetime64[us]
end_utc           datetime64[us, UTC]
lateness_s                    float64
trading_date                   object
bar_time                       object
config_version                    str
dtype: object


Check Nulls in `Exchange_calendar`

In [ ]:
print("Distinct session_open values:")
display(exchange_calendar["session_open"].drop_duplicates().sort_values())

print("Distinct session_close values:")
display(exchange_calendar["session_close"].drop_duplicates().sort_values())

Distinct session_open values:


0     09:30:00
12         NaT
Name: session_open, dtype: object

Distinct session_close values:


75    13:00:00
0     16:00:00
12         NaT
Name: session_close, dtype: object

According to the calendar definition, `session_open` and `session_close` are nullable local session hours, empty when the market is closed.

So the NaN values are valid and expected, not necessarily data quality issues.

In [ ]:
bars_with_runs.head()

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,ts_local,end_utc,lateness_s,trading_date,bar_time,config_version
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0,2025-01-02 09:30:00,2025-01-02 14:31:00+00:00,0.0,2025-01-02,09:30:00,1.4.0
1,RUN-2025A,ARVT,2025-01-02 09:31:00,119.93,119.98,119.35,119.39,45001,2025-01-02 14:32:01+00:00,1,2025-01-02 09:31:00,2025-01-02 14:32:00+00:00,1.0,2025-01-02,09:31:00,1.4.0
2,RUN-2025A,ARVT,2025-01-02 09:32:00,119.39,119.66,119.30,119.53,5825,2025-01-02 14:33:00+00:00,2,2025-01-02 09:32:00,2025-01-02 14:33:00+00:00,0.0,2025-01-02,09:32:00,1.4.0
3,RUN-2025A,ARVT,2025-01-02 09:33:00,119.53,119.57,119.23,119.35,8195,2025-01-02 14:34:00+00:00,3,2025-01-02 09:33:00,2025-01-02 14:34:00+00:00,0.0,2025-01-02,09:33:00,1.4.0
4,RUN-2025A,ARVT,2025-01-02 09:34:00,119.35,119.44,119.33,119.38,5631,2025-01-02 14:35:01+00:00,4,2025-01-02 09:34:00,2025-01-02 14:35:00+00:00,1.0,2025-01-02,09:34:00,1.4.0


In [ ]:
# Check whether date is unique in the exchange calendar
duplicate_dates = exchange_calendar[
    exchange_calendar["date"].duplicated(keep=False)
]

display(duplicate_dates)

,date,is_trading_day,session_open,session_close,timezone,note


In [ ]:
bars_with_calendar = bars_with_runs.merge(
    exchange_calendar,
    left_on="trading_date",
    right_on="date",
    how="left",
    validate="many_to_one"
)

In [ ]:
print("Rows before merge:", len(bars_with_runs))
print("Rows after merge:", len(bars_with_calendar))

assert len(bars_with_calendar) == len(bars_with_runs), \
    "Unexpected row duplication after calendar merge!"

Rows before merge: 4257598
Rows after merge: 4257598


In [ ]:
assert len(bars_with_calendar) == len(bars_with_runs), \
    "Unexpected row duplication after calendar merge!"

In [ ]:
bars_with_calendar

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,...,lateness_s,trading_date,bar_time,config_version,date,is_trading_day,session_open,session_close,timezone,note
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0,...,0.0,2025-01-02,09:30:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN
1,RUN-2025A,ARVT,2025-01-02 09:31:00,119.93,119.98,119.35,119.39,45001,2025-01-02 14:32:01+00:00,1,...,1.0,2025-01-02,09:31:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN
2,RUN-2025A,ARVT,2025-01-02 09:32:00,119.39,119.66,119.30,119.53,5825,2025-01-02 14:33:00+00:00,2,...,0.0,2025-01-02,09:32:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN
3,RUN-2025A,ARVT,2025-01-02 09:33:00,119.53,119.57,119.23,119.35,8195,2025-01-02 14:34:00+00:00,3,...,0.0,2025-01-02,09:33:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN
4,RUN-2025A,ARVT,2025-01-02 09:34:00,119.35,119.44,119.33,119.38,5631,2025-01-02 14:35:01+00:00,4,...,1.0,2025-01-02,09:34:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4257593,RUN-2025C,INDEX,2025-06-30 15:55:00,1234.18,1234.59,1234.17,1234.43,126336,2025-06-30 19:56:00+00:00,4271219,...,0.0,2025-06-30,15:55:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN
4257594,RUN-2025C,INDEX,2025-06-30 15:56:00,1234.43,1235.04,1234.19,1235.00,135129,2025-06-30 19:57:02+00:00,4271220,...,2.0,2025-06-30,15:56:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN
4257595,RUN-2025C,INDEX,2025-06-30 15:57:00,1235.00,1235.18,1234.12,1234.39,126898,2025-06-30 19:58:00+00:00,4271221,...,0.0,2025-06-30,15:57:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN
4257596,RUN-2025C,INDEX,2025-06-30 15:58:00,1234.39,1234.42,1234.13,1234.37,115500,2025-06-30 19:59:02+00:00,4271222,...,2.0,2025-06-30,15:58:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN


In [ ]:
bars_with_calendar["session_status"] = np.where(
    (
        bars_with_calendar["is_trading_day"].eq(True)
    ) &
    (
        bars_with_calendar["bar_time"] >= bars_with_calendar["session_open"]
    ) &
    (
        bars_with_calendar["bar_time"] <= bars_with_calendar["session_close"]
    ),
    "IN_SESSION",
    "OUTSIDE_SESSION"
)

In [ ]:
bars_with_calendar["session_status"].value_counts(dropna=False)

session_status
IN_SESSION         4240875
OUTSIDE_SESSION      16723
Name: count, dtype: int64

In [ ]:
bars_with_calendar

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,...,trading_date,bar_time,config_version,date,is_trading_day,session_open,session_close,timezone,note,session_status
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0,...,2025-01-02,09:30:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
1,RUN-2025A,ARVT,2025-01-02 09:31:00,119.93,119.98,119.35,119.39,45001,2025-01-02 14:32:01+00:00,1,...,2025-01-02,09:31:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
2,RUN-2025A,ARVT,2025-01-02 09:32:00,119.39,119.66,119.30,119.53,5825,2025-01-02 14:33:00+00:00,2,...,2025-01-02,09:32:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
3,RUN-2025A,ARVT,2025-01-02 09:33:00,119.53,119.57,119.23,119.35,8195,2025-01-02 14:34:00+00:00,3,...,2025-01-02,09:33:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
4,RUN-2025A,ARVT,2025-01-02 09:34:00,119.35,119.44,119.33,119.38,5631,2025-01-02 14:35:01+00:00,4,...,2025-01-02,09:34:00,1.4.0,2025-01-02,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4257593,RUN-2025C,INDEX,2025-06-30 15:55:00,1234.18,1234.59,1234.17,1234.43,126336,2025-06-30 19:56:00+00:00,4271219,...,2025-06-30,15:55:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
4257594,RUN-2025C,INDEX,2025-06-30 15:56:00,1234.43,1235.04,1234.19,1235.00,135129,2025-06-30 19:57:02+00:00,4271220,...,2025-06-30,15:56:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
4257595,RUN-2025C,INDEX,2025-06-30 15:57:00,1235.00,1235.18,1234.12,1234.39,126898,2025-06-30 19:58:00+00:00,4271221,...,2025-06-30,15:57:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
4257596,RUN-2025C,INDEX,2025-06-30 15:58:00,1234.39,1234.42,1234.13,1234.37,115500,2025-06-30 19:59:02+00:00,4271222,...,2025-06-30,15:58:00,1.4.0,2025-06-30,True,09:30:00,16:00:00,America/New_York,NaN,IN_SESSION
